# m14-Schwarz: cluster-localized refinement prototype

**Problem.** Global m14 processes every corner of the field, even when
only ~5% of cells are folded. On a 320×456 slice with sparse fold
clusters this is mostly wasted compute — the optimizer is iterating on
cells that were already at the anchor and already feasible.

**Idea.** Detect fold clusters first (connected components of
`min(T1, T2) <= 0`, dilated by `merge_dilation=2` to group nearby
clusters). For each cluster, extract a bounding-box crop with `pad`
cells of context, run global m14 on the crop, splice back. Schwarz-style
domain decomposition — each subproblem is tiny, independent of slice
size.

If splicing introduces new folds at crop boundaries, do another round.
After `max_outer_iters` rounds, fall back to global m14 if there's
still progress to be had. Also fall back immediately when a single
cluster spans most of the field (no point in cropping).

**This notebook.**

1. Run the prototype on five cases ranging from very sparse (3 separated
   clusters on a 30×30 synthetic field) to fully saturated (B0039 z=12
   full slice with 8978 folds).
2. Compare wall clock, L1, L2 against global m14.
3. Visualize the fold-cluster decomposition + the per-cluster bounding
   boxes for a representative case.


In [ ]:
import os, sys, time, contextlib, io, warnings
sys.path.insert(0, os.path.abspath('../..'))
sys.path.insert(0, os.path.abspath('.'))
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label as cc_label, binary_dilation, generate_binary_structure

from dvfopt.core.wallbreakers import iterative_2d_tri_refine_repair
from dvfopt.jacobian.triangle_sign import _triangle_areas_2d

from _m14_schwarz_proto import m14_schwarz, _stats, _fold_clusters

THRESHOLD = 0.01


def _silent(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)


def _plant_fold(arr, cy, cx, amp=0.8):
    arr[cy, cx] += amp
    arr[cy+1, cx] -= amp
    arr[cy, cx+1] -= amp
    arr[cy+1, cx+1] += amp


def load_b0039(cy, cx, size):
    arr = np.load('../../data/dvfs/b0039/b0039_laplacian_deformation_field.npy')
    dy = arr[1, 12, cy:cy+size, cx:cx+size].astype(np.float64).copy()
    dx = arr[2, 12, cy:cy+size, cx:cx+size].astype(np.float64).copy()
    return np.stack([dy, dx])


def load_b0039_full():
    arr = np.load('../../data/dvfs/b0039/b0039_laplacian_deformation_field.npy')
    return np.stack([arr[1, 12].astype(np.float64).copy(),
                     arr[2, 12].astype(np.float64).copy()])


## 1. Test cases

Spans the full density spectrum:

| case | description | density |
|---|---|---|
| `synth_30x30_sparse_8` | 30×30 synthetic, 3 separated fold cores | very sparse |
| `z12_30x30_379` | B0039 z=12 crop (cy=120, cx=180) | moderate-dense |
| `z12_30x30_1484` | B0039 z=12 crop (cy=180, cx=180) | near-saturated |
| `z12_60x60` | B0039 z=12 crop (cy=140, cx=160) | multi-cluster real data |
| `z12_full_320x456` | B0039 z=12 full slice (8978 folds) | the wall |


In [ ]:
def synth_sparse(seed=0):
    np.random.seed(seed)
    H, W = 30, 30
    dy = np.random.normal(0, 0.05, (H, W))
    dx = np.random.normal(0, 0.05, (H, W))
    _plant_fold(dx, 5, 5)
    _plant_fold(dx, 5, 20)
    _plant_fold(dy, 22, 12)
    return np.stack([dy, dx])


CASES = [
    ('synth_30x30_sparse_8', synth_sparse(), dict()),
    ('z12_30x30_379',        load_b0039(120, 180, 30), dict()),
    ('z12_30x30_1484',       load_b0039(180, 180, 30), dict()),
    ('z12_60x60',            load_b0039(140, 160, 60), dict()),
    ('z12_full_320x456',     load_b0039_full(), dict(time_budget_s=900.0)),
]
for label, phi, _ in CASES:
    H, W = phi.shape[1], phi.shape[2]
    n_neg, min_T = _stats(phi)
    print(f'{label:<22} {H}x{W:<8}  init n_neg={n_neg:>5}  init min_T={min_T:>+8.3f}')


## 2. Benchmark: m14-Schwarz vs global m14

Both run with `anchor='l1'` and `threshold=0.01`. The m14-Schwarz
prototype uses `pad=4`, `merge_dilation=2`, `max_outer_iters=3`,
`fallback_size_ratio=0.7`.


In [ ]:
def run_one(method_label, fn, phi_in, **kwargs):
    t0 = time.time()
    extra = {}
    try:
        if method_label == 'm14_schwarz':
            phi_out, hist = fn(phi_in.copy(), threshold=THRESHOLD,
                                anchor='l1', verbose=0, record_history=True,
                                **kwargs)
            extra = dict(
                fallback=hist.get('fallback_to_global', False),
                n_clusters=len(hist.get('cluster_runs', [])),
                outer_rounds=len(hist.get('outer_rounds', [])),
            )
        else:
            phi_out = _silent(fn, phi_in.copy(),
                              threshold=THRESHOLD, anchor='l1', verbose=0,
                              **kwargs)
    except Exception as exc:
        return dict(method=method_label, wall_s=time.time()-t0,
                    error=f'{type(exc).__name__}: {exc}', feasible=False,
                    n_neg=-1, min_T=float('nan'), L1=float('nan'),
                    L2=float('nan'), **extra)
    wall = time.time() - t0
    n_neg, min_T = _stats(phi_out)
    diff = (phi_out - phi_in).ravel()
    return dict(
        method=method_label, wall_s=wall,
        n_neg=n_neg, min_T=min_T,
        L1=float(np.abs(diff).sum()),
        L2=float(np.sqrt(np.dot(diff, diff))),
        feasible=(n_neg == 0 and min_T >= THRESHOLD - 1e-5),
        error='', **extra,
    )


In [ ]:
rows = []
for label, phi, kw in CASES:
    print(f'\n=== {label} ===')
    for method_label, fn in [
        ('m14_global',  iterative_2d_tri_refine_repair),
        ('m14_schwarz', m14_schwarz),
    ]:
        r = run_one(method_label, fn, phi, **kw)
        r['case'] = label
        rows.append(r)
        extras = ''
        if 'fallback' in r:
            extras = f"  fallback={r['fallback']}  clusters={r['n_clusters']}"
        tag = 'OK' if r['feasible'] else ('ERR' if r['error'] else 'FAIL')
        print(f'  [{tag:>4}] {method_label:<14}  wall={r["wall_s"]:>7.2f}s  '
              f'n_neg={r["n_neg"]:>5}  min_T={r["min_T"]:+.4f}  '
              f'L1={r["L1"]:>9.1f}  L2={r["L2"]:>8.2f}'
              + extras)


## 3. Summary


In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
df_summary = df.pivot_table(index='case', columns='method',
                              values=['wall_s', 'L1', 'L2', 'feasible'],
                              aggfunc='first')
print('=== Speedup + L1 ratio ===')
for label, _, _ in CASES:
    g = next((r for r in rows if r['case'] == label
              and r['method'] == 'm14_global'), None)
    s = next((r for r in rows if r['case'] == label
              and r['method'] == 'm14_schwarz'), None)
    if g is None or s is None:
        continue
    sp = g['wall_s'] / s['wall_s'] if s['wall_s'] > 0 else float('inf')
    l1_ratio = s['L1'] / g['L1'] if g['L1'] > 0 else float('nan')
    fb = '(fallback)' if s.get('fallback') else ''
    print(f'  {label:<22}  speedup={sp:>5.2f}x  L1 ratio={l1_ratio:>5.2f}x  {fb}')
df


## 4. Visualization: fold-cluster decomposition + bounding boxes

For the multi-cluster real-data case (`z12_60x60`), show:

- Left: initial `min(T1, T2)` heatmap with fold cells outlined.
- Center: cluster decomposition (each cluster a different color) with
  bounding-box overlays for the crops m14-Schwarz processes.
- Right: final state after m14-Schwarz.

If clusters are well-separated, the crops cover only a small fraction
of the slice — that's where the wall-clock win comes from.


In [ ]:
from matplotlib.patches import Rectangle

phi_viz = load_b0039(140, 160, 60)
T1, T2 = _triangle_areas_2d(phi_viz[0], phi_viz[1])
min_T = np.minimum(T1, T2)

bboxes, fold_mask = _fold_clusters(phi_viz, merge_dilation=2)
n_clusters = len(bboxes)

# Run m14-schwarz with viz pad=4 for the boxes.
phi_after, hist = m14_schwarz(phi_viz.copy(), threshold=THRESHOLD,
                                anchor='l1', pad=4, merge_dilation=2,
                                verbose=0, record_history=True)
T1a, T2a = _triangle_areas_2d(phi_after[0], phi_after[1])
min_T_after = np.minimum(T1a, T2a)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

vm = max(abs(min_T.min()), 0.05)
im0 = axes[0].imshow(min_T, cmap='RdBu_r', vmin=-vm, vmax=vm)
axes[0].set_title(f'before: n_folds={int(fold_mask.sum())}', fontsize=10)
axes[0].set_xticks([]); axes[0].set_yticks([])
fig.colorbar(im0, ax=axes[0], shrink=0.85)

# Cluster labels.
grouped = binary_dilation(fold_mask, iterations=2)
labels_arr, _ = cc_label(grouped, structure=generate_binary_structure(2, 2))
axes[1].imshow(labels_arr, cmap='tab20')
for b in bboxes:
    H, W = phi_viz.shape[1], phi_viz.shape[2]
    pad = 4
    y0 = max(0, b['cy0'] - pad)
    y1 = min(H, b['cy1'] + pad + 2)
    x0 = max(0, b['cx0'] - pad)
    x1 = min(W, b['cx1'] + pad + 2)
    rect = Rectangle((x0 - 0.5, y0 - 0.5), (x1 - x0), (y1 - y0),
                     fill=False, edgecolor='red', linewidth=1.5)
    axes[1].add_patch(rect)
axes[1].set_title(f'{n_clusters} clusters + bounding boxes', fontsize=10)
axes[1].set_xticks([]); axes[1].set_yticks([])

vm_a = max(abs(min_T_after.min()), 0.05)
im2 = axes[2].imshow(min_T_after, cmap='RdBu_r', vmin=-vm_a, vmax=vm_a)
axes[2].set_title(f'after m14-Schwarz: n_folds={int((min_T_after<=0).sum())}',
                  fontsize=10)
axes[2].set_xticks([]); axes[2].set_yticks([])
fig.colorbar(im2, ax=axes[2], shrink=0.85)

plt.suptitle('B0039 z=12 60x60 crop: cluster decomposition')
plt.show()


## Findings

Fill in after the benchmark completes. Expected pattern:

- **Sparse synthetic / multi-cluster real data:** big speedup
  (5–20×) because each cluster is small relative to the full field.
  L1 should be ≤ global m14 (or close, since each cluster sees only
  its own anchor neighborhood).
- **Single-large-cluster cases (`z12_30x30_*` and `z12_full`):**
  the `fallback_size_ratio` triggers and we fall back to global m14.
  Wall-clock is then `global m14 wall + cluster-detection overhead`
  — essentially unchanged.

If sparse cases show the expected speedup, the prototype is worth
promoting into the public surface (as `iterative_2d_tri_refine_repair`
with an optional `schwarz=True` flag or a new
`iterative_2d_tri_schwarz_refine` entry point).
